# 00 — Environment check

Records the hardware and software of the machine that runs the re-analysis, so that every result in this repository can be traced to a known setup.
Output: `results/environment_report.json`.

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import psutil

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
(ROOT / "results").mkdir(exist_ok=True)

def run(cmd):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        return (out.stdout + out.stderr).strip().splitlines()[0]
    except Exception as exc:
        return f"not available ({type(exc).__name__})"

report = {
    "machine": platform.node(),
    "os": f"{platform.system()} {platform.release()} ({platform.version()})",
    "cpu": platform.processor(),
    "physical_cores": psutil.cpu_count(logical=False),
    "logical_cores": psutil.cpu_count(logical=True),
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "python": sys.version.split()[0],
    "java": run(["java", "-version"]),
    "git": run(["git", "--version"]),
    "disks": {
        p.device: {
            "total_gb": round(shutil.disk_usage(p.mountpoint).total / 1e9),
            "free_gb": round(shutil.disk_usage(p.mountpoint).free / 1e9),
        }
        for p in psutil.disk_partitions() if p.fstype
    },
    "project_root": str(ROOT),
    "project_disk_free_gb": round(shutil.disk_usage(ROOT).free / 1e9),
}
(ROOT / "results" / "environment_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

## Requirements for the full re-analysis

| Resource | Needed | Why |
|---|---|---|
| Free disk | about 300 GB | 96 Thermo `.raw` files (202 GB) plus search results |
| RAM | 32 GB or more | MSFragger index for a human database with TMT modifications |
| Java | 11 or newer | FragPipe / MSFragger |

In [ ]:
need_disk, need_ram = 300, 32
print("disk:", "OK" if report["project_disk_free_gb"] >= need_disk else f"too little ({report['project_disk_free_gb']} GB free, {need_disk} GB needed)")
print("RAM :", "OK" if report["ram_gb"] >= need_ram else f"too little ({report['ram_gb']} GB, {need_ram} GB needed)")